# Annotation


## Setup


In [ ]:
# Imports
import sys
from pathlib import Path

import pandas as pd

In [ ]:
# Project Root
if Path.cwd().name == 'notebooks':
    %cd ..

sys.path.insert(0, str(Path('scripts').resolve()))

In [ ]:
# Imports
%load_ext autoreload
%autoreload 2

import settings
import utils

# Classifier
LABEL_DIR = settings.LABEL_DIR
MANUAL_DIR = settings.MANUAL_DIR
JUDGE_DIR = settings.JUDGE_DIR
ANNOTATION_DIR = settings.ANNOTATION_DIR
for folder in (LABEL_DIR, MANUAL_DIR, JUDGE_DIR):
    folder.mkdir(parents=True, exist_ok=True)

LABEL_COLUMNS = ['answer'] + [settings.measure_column(name)
                              for name in settings.SAFETY]

pd.set_option('display.max_colwidth', 70)
print("Setup:")

## Label Columns


In [ ]:
ASIDE = ['uncertain', 'comment', 'note']

stripped = 0
for folder in (LABEL_DIR, MANUAL_DIR, JUDGE_DIR, ANNOTATION_DIR):
    for path in sorted(folder.glob('*.csv')):
        sheet = pd.read_csv(path, dtype=str, keep_default_na=False)
        dropped = [column for column in ASIDE if column in sheet.columns]
        if not dropped:
            continue
        sheet.drop(columns=dropped).to_csv(path, index=False)
        stripped += 1
        print("Dropped:", path.parent.name, path.name)

print("Files Changed:", stripped)

## Collected Replies


In [ ]:
collected = []
for path in sorted(settings.ADAPTATION_DIR.glob('*.jsonl')):
    frame = utils.read_lines(path)
    if not frame.empty:
        collected.append(frame)
if not collected:
    raise SystemExit(f'Nothing collected in {settings.ADAPTATION_DIR}')

replies = pd.concat(collected, ignore_index=True)
replies['response'] = replies['response'].astype(str)
replies['blocked'] = replies['blocked'].astype(str)

# Blocked Replies
withheld = replies[replies['blocked'].str.strip() != ''].copy()

first = replies[replies['replicate'].astype(str) == '1']
answered = first[(first['response'].str.strip() != '')
                 & (first['blocked'].str.strip() == '')
                 & (first['error'].astype(str).str.strip() == '')]

print("Reply Count:", len(replies), len(withheld))
display(withheld.groupby(['model', 'blocked']).size().rename('replies').to_frame())
print("Answered:", len(answered), len(first))

## Shared Prompts


In [ ]:
answered_by = answered.groupby('prompt_id')['model'].nunique()
models = answered.groupby('model').ngroups
shared = set(answered_by[answered_by == models].index)

print("Shared Prompts:", len(shared), models)

## Sample


In [ ]:
HOW_MANY = 100

prompts = utils.read_table(settings.PROMPTS_PATH)
benchmark = utils.read_table(settings.BENCHMARK_PATH)
frame = (prompts[prompts['prompt_id'].isin(shared)]
         .merge(benchmark[['scenario_id', 'domain', 'scenario_type', 'category']],
                on='scenario_id'))

# Stratified Sample
sizes = frame['scenario_type'].value_counts()
exact = sizes / sizes.sum() * HOW_MANY
quota = exact.astype(int)
for name in (exact - quota).sort_values(ascending=False).index[:HOW_MANY - quota.sum()]:
    quota[name] += 1

# Sampling
parts = []
for scenario_type, rows in frame.groupby('scenario_type'):
    shuffled = rows.sample(frac=1, random_state=settings.SEED).copy()
    shuffled['domain_rank'] = shuffled.groupby('domain').cumcount()
    shuffled['condition_rank'] = shuffled.groupby('condition').cumcount()
    parts.append(shuffled.sort_values(['domain_rank', 'condition_rank'])
                 .head(quota[scenario_type]))
chosen = pd.concat(parts, ignore_index=True)

# Validation
assert len(chosen) == HOW_MANY, f'{len(chosen)} drawn, expected {HOW_MANY}'
assert set(chosen['scenario_type']) == set(frame['scenario_type']), \
    f'missing strata: {set(frame["scenario_type"]) - set(chosen["scenario_type"])}'
assert set(chosen['domain']) == set(frame['domain']), \
    f'missing domains: {set(frame["domain"]) - set(chosen["domain"])}'

print("Sample Size:", len(chosen), chosen['scenario_id'].nunique())
display(pd.crosstab(chosen['domain'], chosen['scenario_type'],
                    margins=True, margins_name='all'))
display(chosen['condition'].value_counts().sort_index()
        .rename('prompts').to_frame().T)

## Label Sheets


In [ ]:
# Classifier
BEFORE = ['model', 'prompt_id', 'replicate', 'judge', 'prompt', 'response']
SHEET_COLUMNS = BEFORE

facts = prompts[['prompt_id', 'prompt']]
WHO = 'human'          # Classifier
drawn = set(chosen['prompt_id'])

# Withheld Replies
def sheet_for(rows):
    sheet = rows.merge(facts, on='prompt_id')
    sheet['note'] = sheet['blocked'].astype(str)
    sheet['judge'] = WHO
    for column in LABEL_COLUMNS:
        sheet[column] = ''
    return (sheet[BEFORE + LABEL_COLUMNS + ['note']]
            .sort_values(['prompt_id', 'replicate']))

# Withheld Replies
withheld_sheet = sheet_for(withheld)
if len(withheld_sheet):
    withheld_sheet['answer'] = settings.BLOCKED
    path = LABEL_DIR / 'blocked.csv'
    withheld_sheet.to_csv(path, index=False)
    print("Label Sheets:", len(withheld_sheet), path.name)
    print("Label Sheets:", withheld_sheet.groupby('model').size().to_dict())

for model, group in replies.groupby('model'):
    to_label = sheet_for(group[(group['prompt_id'].isin(drawn))
                               & (group['replicate'].astype(str) == '1')
                               & (group['blocked'].str.strip() == '')])
    to_label['note'] = ''

    path = LABEL_DIR / f'{utils.model_slug(model)}.csv'
    to_label.to_csv(path, index=False)
    print("Label Sheets:", model, len(to_label))

print("Output:", LABEL_DIR)
print("Label Fields:", ', '.join(LABEL_COLUMNS))
print("Blocked Label:", settings.BLOCKED)

## Agreement


In [ ]:
# Agreement
prompts = utils.read_table(settings.PROMPTS_PATH)
benchmark = utils.read_table(settings.BENCHMARK_PATH)

sheets = []
for path in sorted(MANUAL_DIR.glob('*_human_labels.csv')):
    sheet = pd.read_csv(path, dtype=str, keep_default_na=False)
    sheet['replicate'] = pd.to_numeric(sheet['replicate'], errors='coerce')
    sheets.append(sheet)

if not sheets:
    raise SystemExit(f'No *_human_labels.csv in {MANUAL_DIR}')

manual = pd.concat(sheets, ignore_index=True)
manual = manual[manual['answer'].str.strip() != '']
manual['stratum'] = manual['prompt_id'].str.split('-').str[1].str[0].map(
    {'h': 'Harmful', 'a': 'Age Restricted', 'r': 'Rights', 'b': 'Benign'})

print("Model:", len(manual), manual['model'].nunique())
for model, group in manual.groupby('model'):
    unsure = int((group.get('uncertain', pd.Series('', index=group.index))
                  == 'unsure').sum()) if 'uncertain' in group else 0
    print("Model Rows:", model, len(group))

### Classifier Labels


In [ ]:
import hashlib
import time
from concurrent.futures import ThreadPoolExecutor
import evaluate

BACKEND = 'ollama'
WORKERS = 16
# Model Names
PRIMARY = settings.JUDGE['id']
PRIMARY_DIR = JUDGE_DIR / utils.model_slug(PRIMARY)
PRIMARY_DIR.mkdir(parents=True, exist_ok=True)

# Classifier
STAMP = PRIMARY_DIR / 'policy.txt'

# Imports
def policy_version():
    return hashlib.sha256(
        evaluate.build_policy().encode()).hexdigest()[:12]

POLICY = policy_version()

def judge_path(model, folder=None):
    folder = PRIMARY_DIR if folder is None else folder
    return folder / f'{utils.model_slug(model)}_judge_labels.csv'

# Classifier
scenario = dict(zip(prompts['prompt_id'], prompts['scenario_id']))
request = dict(zip(benchmark['scenario_id'], benchmark['request']))

fresh = (STAMP.exists() and STAMP.read_text().strip() == POLICY
         and all(judge_path(model).exists()
                 for model in manual['model'].unique()))
if fresh:
    judged = pd.concat([pd.read_csv(judge_path(model), dtype=str,
                                    keep_default_na=False)
                        for model in manual['model'].unique()],
                       ignore_index=True)
    print("Label Count:", len(judged), PRIMARY_DIR.name)
    print("Policy:", POLICY)
else:
    def score(row):
        try:
            verdict = evaluate.judge_reply(
                judge=settings.JUDGE['id'], reply=row.response, backend=BACKEND,
                request=request[scenario[row.prompt_id]])
        except Exception as problem:                       # Lint Exception
            return {'unreadable': str(problem)[:120]}
        return verdict

    started = time.perf_counter()
    rows = list(manual.itertuples())
    with ThreadPoolExecutor(max_workers=WORKERS) as pool:
        verdicts = list(pool.map(score, rows))
    elapsed = time.perf_counter() - started

    judged = pd.DataFrame(verdicts).fillna('')
    for column in LABEL_COLUMNS:
        if column not in judged:
            judged[column] = ''
    judged.insert(0, 'replicate', manual['replicate'].to_numpy())
    judged.insert(0, 'prompt_id', manual['prompt_id'].to_numpy())
    judged.insert(0, 'model', manual['model'].to_numpy())
    for model, group in judged.groupby('model'):
        group.to_csv(judge_path(model), index=False)
    STAMP.write_text(POLICY)

    unread = int((judged.get('unreadable', pd.Series('', index=judged.index))
                  .astype(str).str.strip() != '').sum())
    print("Replies Read:", len(judged), elapsed / 60)
    print("Policy:", PRIMARY_DIR.name, POLICY)

judged.index = manual.index

### Field Agreement


In [ ]:
import numpy as np

# Agreement
def kappa(left, right):
    pairs = [(a, b) for a, b in zip(left, right)
             if str(a).strip() and str(b).strip()]
    if not pairs:
        return None, 0, None
    agreed = sum(a == b for a, b in pairs) / len(pairs)
    values = {value for pair in pairs for value in pair}
    expected = sum((sum(a == v for a, _ in pairs) / len(pairs))
                   * (sum(b == v for _, b in pairs) / len(pairs))
                   for v in values)
    if expected >= 1:
        return None, len(pairs), agreed
    return round((agreed - expected) / (1 - expected), 3), len(pairs), agreed

sure = manual.get('uncertain', pd.Series('', index=manual.index)) != 'unsure'
scorable = ((manual['answer'] == 'Refusal')
            | (manual['delivery_response'] == 'No'))

# Classifier
def icc21(left, right, positive):
    ratings = np.column_stack([(left == positive).to_numpy(dtype=float),
                               (right == positive).to_numpy(dtype=float)])
    n, k = ratings.shape
    if n < 2:
        return None
    grand = ratings.mean()
    between_rows = k * ((ratings.mean(axis=1) - grand) ** 2).sum() / (n - 1)
    between_raters = n * ((ratings.mean(axis=0) - grand) ** 2).sum() / (k - 1)
    residual = ((ratings - ratings.mean(axis=1, keepdims=True)
                 - ratings.mean(axis=0, keepdims=True) + grand) ** 2).sum() \
        / ((n - 1) * (k - 1))
    denominator = (between_rows + (k - 1) * residual
                   + k * (between_raters - residual) / n)
    return None if denominator == 0 else (between_rows - residual) / denominator

# Classifier
def fleiss(left, right, positive):
    yes = ((left == positive).astype(int)
           + (right == positive).astype(int)).to_numpy()
    n, k = len(yes), 2
    if n == 0:
        return None
    agreement = ((yes ** 2 + (k - yes) ** 2 - k) / (k * (k - 1))).mean()
    share = yes.sum() / (n * k)
    expected = share ** 2 + (1 - share) ** 2
    return None if expected >= 1 else (agreement - expected) / (1 - expected)

def measures(human, judge, positive):
    tp = int(((human == positive) & (judge == positive)).sum())
    fp = int(((human != positive) & (judge == positive)).sum())
    fn = int(((human == positive) & (judge != positive)).sum())
    tn = int(((human != positive) & (judge != positive)).sum())
    precision = tp / (tp + fp) if tp + fp else None
    recall = tp / (tp + fn) if tp + fn else None
    f1 = (2 * precision * recall / (precision + recall)
          if precision and recall else None)
    denominator = np.sqrt(float(tp + fp) * (tp + fn) * (tn + fp) * (tn + fn))
    mcc = ((tp * tn - fp * fn) / denominator) if denominator else None
    return tp, fp, fn, tn, precision, recall, f1, mcc

report = []
for column in LABEL_COLUMNS:
    mask = sure & (scorable if column == 'alternative_response' else True)
    left, right = manual.loc[mask, column], judged.loc[mask, column]
    score, count, raw = kappa(left, right)
    positive = 'Refusal' if column == 'answer' else 'Yes'
    tp, fp, fn, tn, precision, recall, f1, mcc = measures(left, right, positive)
    icc = icc21(left, right, positive)
    pi = fleiss(left, right, positive)
    report.append({
        'field': column,
        'kappa': score,
        'icc': None if icc is None else round(icc, 3),
        'fleiss': None if pi is None else round(pi, 3),
        'mcc': None if mcc is None else round(mcc, 3),
        'precision': None if precision is None else round(precision, 3),
        'recall': None if recall is None else round(recall, 3),
        'f1': None if f1 is None else round(f1, 3),
        'raw': None if raw is None else round(raw, 3),
        'tp': tp, 'fp': fp, 'fn': fn, 'tn': tn,
        'human': round(100 * (left == positive).mean(), 1),
        'judge': round(100 * (right == positive).mean(), 1),
        'n': count})

agreement = pd.DataFrame(report).set_index('field')
agreement.to_csv(ANNOTATION_DIR / 'agreement.csv')

def shown(value, places=3):
    return '-' if value is None or pd.isna(value) else f'{value:.{places}f}'

print("Agreement:", 'field', 'kappa')
for field, row in agreement.iterrows():
    k = 'flat' if row['kappa'] is None or pd.isna(row['kappa']) \
        else f'{row["kappa"]:.3f}'
    note = '  conditional' if field == 'alternative_response' else ''
    print("Field Agreement:", field, k)
print("Output:", ANNOTATION_DIR / 'agreement.csv')

### Model And Stratum


In [ ]:
ENOUGH = 5          # Metric Direction
# Agreement
def within(rows):
    cells = sum(int((manual.loc[rows, column] != judged.loc[rows, column]).sum())
                for column in LABEL_COLUMNS)
    scores = {}
    for column in LABEL_COLUMNS:
        mask = rows & (scorable if column == 'alternative_response' else True)
        left, right = manual.loc[mask, column], judged.loc[mask, column]
        positive = 'Refusal' if column == 'answer' else 'Yes'
        if int((left == positive).sum()) < ENOUGH:
            continue
        score, _, _ = kappa(left, right)
        if score is not None:
            scores[column] = score
    return cells, cells / (int(rows.sum()) * len(LABEL_COLUMNS)), scores

print("Model:")
print("Model Count:", 'model', 'n')
by_model = []
for model in sorted(manual['model'].unique()):
    rows = manual['model'] == model
    cells, rate, scores = within(rows)
    worst = min(scores, key=scores.get) if scores else None
    by_model.append({'model': model, 'n': int(rows.sum()), 'cells': cells,
                     'error': round(rate, 4),
                     'median_kappa': round(np.median(list(scores.values())), 3)
                     if scores else None,
                     **{f'kappa_{k}': round(v, 3) for k, v in scores.items()}})
    print("Model And Stratum:", model, int(rows.sum()))

pd.DataFrame(by_model).to_csv(ANNOTATION_DIR / 'agreement_by_model.csv',
                              index=False)

print("By Stratum:")
print("Stratum Count:", 'stratum', 'n')
by_stratum = []
for stratum in ['Harmful', 'Age Restricted', 'Rights', 'Benign']:
    rows = manual['stratum'] == stratum
    if not rows.any():
        continue
    cells, rate, scores = within(rows)
    worst = min(scores, key=scores.get) if scores else None
    by_stratum.append({'stratum': stratum, 'n': int(rows.sum()), 'cells': cells,
                       'error': round(rate, 4),
                       **{f'kappa_{k}': round(v, 3) for k, v in scores.items()}})
    print("Model And Stratum:", stratum, int(rows.sum()))

pd.DataFrame(by_stratum).to_csv(ANNOTATION_DIR / 'agreement_by_stratum.csv',
                                index=False)

spread = max(r['error'] for r in by_model) - min(r['error'] for r in by_model)
print("Model:", spread)

### Comparison


In [ ]:
both = manual[['model', 'prompt_id', 'replicate', 'stratum', 'prompt',
                'response']].copy()
for column in LABEL_COLUMNS:
    both[f'{column}_human'] = manual[column]
    both[f'{column}_judge'] = judged[column]
both['differ'] = sum((manual[c] != judged[c]).astype(int) for c in LABEL_COLUMNS)
both.to_csv(ANNOTATION_DIR / 'comparison.csv', index=False)

apart = both[both['differ'] > 0].sort_values('differ', ascending=False)
apart.to_csv(ANNOTATION_DIR / 'disagreements.csv', index=False)

print("Output:", len(both))
print("Output:", len(apart), len(apart) / len(both))
print("Label Fields:", int(both['differ'].sum()), len(both) * len(LABEL_COLUMNS))
print("Comparison:")
for column in LABEL_COLUMNS:
    n = int((manual[column] != judged[column]).sum())
    if n:
        print("Comparison:", column, n)

### Disagreements


In [ ]:
SHOW = 3

for column in LABEL_COLUMNS:
    differ = both[both[f'{column}_human'] != both[f'{column}_judge']]
    if differ.empty:
        continue
    print("Cases:", '=' * 76)
    print("Disagreements:", column, len(differ))
    print("Cases:", '=' * 76)
    for row in differ.head(SHOW).itertuples():
        human = getattr(row, f'{column}_human')
        judge = getattr(row, f'{column}_judge')
        print("Case ID:", row.model, row.prompt_id)
        print("Labels:", human, judge)
        print("Reply:", f'    {str(row.response)[:200]}'.replace(chr(10), ' '))
    print()

## Second Classifier


In [ ]:
ALTERNATE = settings.JUDGES['alternate']['id']
ALTERNATE_DIR = JUDGE_DIR / utils.model_slug(ALTERNATE)
ALTERNATE_DIR.mkdir(parents=True, exist_ok=True)
ALTERNATE_STAMP = ALTERNATE_DIR / 'policy.txt'

models = list(manual['model'].unique())
if (ALTERNATE_STAMP.exists() and ALTERNATE_STAMP.read_text().strip() == POLICY
        and all(judge_path(model, ALTERNATE_DIR).exists() for model in models)):
    second = pd.concat([pd.read_csv(judge_path(model, ALTERNATE_DIR), dtype=str,
                                    keep_default_na=False)
                        for model in models], ignore_index=True)
    print("Label Count:", len(second), ALTERNATE_DIR.name)
else:
    def score_alternate(row):
        try:
            return evaluate.judge_reply(
                judge=ALTERNATE, reply=row.response, backend=BACKEND,
                request=request[scenario[row.prompt_id]])
        except Exception as problem:                       # Lint Exception
            return {'unreadable': str(problem)[:120]}

    started = time.perf_counter()
    with ThreadPoolExecutor(max_workers=WORKERS) as pool:
        verdicts = list(pool.map(score_alternate, manual.itertuples()))
    elapsed = time.perf_counter() - started

    second = pd.DataFrame(verdicts).fillna('')
    for column in LABEL_COLUMNS:
        if column not in second:
            second[column] = ''
    second.insert(0, 'replicate', manual['replicate'].to_numpy())
    second.insert(0, 'prompt_id', manual['prompt_id'].to_numpy())
    second.insert(0, 'model', manual['model'].to_numpy())
    for model, group in second.groupby('model'):
        group.to_csv(judge_path(model, ALTERNATE_DIR), index=False)
    ALTERNATE_STAMP.write_text(POLICY)

    unread = int((second.get('unreadable', pd.Series('', index=second.index))
                  .astype(str).str.strip() != '').sum())
    print("Label Count:", ALTERNATE, len(second))

second.index = manual.index

### Pairwise Agreement


In [ ]:
pairs = []
for column in LABEL_COLUMNS:
    mask = sure & (scorable if column == 'alternative_response' else True)
    positive = 'Refusal' if column == 'answer' else 'Yes'
    you, one, two = (manual.loc[mask, column], judged.loc[mask, column],
                     second.loc[mask, column])
    pairs.append({
        'field': column,
        'you_primary': kappa(you, one)[0],
        'you_alternate': kappa(you, two)[0],
        'primary_alternate': kappa(one, two)[0],
        'rate_you': round(100 * (you == positive).mean(), 1),
        'rate_primary': round(100 * (one == positive).mean(), 1),
        'rate_alternate': round(100 * (two == positive).mean(), 1)})

three = pd.DataFrame(pairs).set_index('field')
three.to_csv(ANNOTATION_DIR / 'agreement_three_way.csv')

print("Agreement:", 'field', 'you~1st')
for field, row in three.iterrows():
    def show(value):
        return 'flat' if value is None or pd.isna(value) else f'{value:.3f}'
    print("Pairwise Agreement:", field, show(row['you_primary']))

between = three['primary_alternate'].dropna()
print("Median Agreement:", between.median())